## Model Categories

#### Ultra Light: 0B-1B
1) https://huggingface.co/JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const_0.3 (0.6B)
2) https://huggingface.co/KingNish/Reasoning-0.5b (0.5B)
3) https://huggingface.co/oopere/Llama-3.2-1B-pruned-40pct (0.9B)
#### Small:  2B-3B
1) https://huggingface.co/Alyon-AI/2_PRYMMAL-ECE-2B-SLERP-V1 (3B)
2) https://huggingface.co/google/gemma-2-2b-it?library=transformers (3B)
3) https://huggingface.co/SpaceYL/ECE_Poirot (2B)
#### Medium: 3B-7B
1) https://huggingface.co/OpenBuddy/openbuddy-zero-3b-v21.2-32k (5B)
2) https://huggingface.co/jpacifico/Distilucie-7B-Math-Instruct-DPO-v0.1 (7B)
3) https://huggingface.co/weathermanj/Menda-3b-Optim-100 (3B)
#### Large: 7B-10B
1) https://huggingface.co/grimjim/Magnolia-v3-Gemma2-8k-9B (8B)
2) https://huggingface.co/marcuscedricridia/cursa-o1-7b-v1.2-normalize-false?library=transformers
3) https://huggingface.co/marcuscedricridia/Abus-7B-Instruct (8B)
#### Massive: 10 or higher
1) [https://huggingface.co/MaziyarPanahi/calme-3.2-instruct-78b (78B)](https://huggingface.co/marcuscedricridia/Abus-7B-Instruct) (10B)
2) https://huggingface.co/arcee-ai/Virtuoso-Lite (10B)
3) https://huggingface.co/Aashraf995/Gemma-Evo-10B (10B)

## Prompt Types

1) Text generation
2) Classification
3) Summarization
4) Prediction
5) Extraction

## Evaluation Prompts

#### Text Generation
1) What are the legal moves for a knight in Chess?
2) What shoes would you recommend for a beginner runner?
3) What is the easiest sweet to bake?
4) What is the bar scene like in Champaign-Urbana?
5) What position should I play in basketball?

#### Classification
1) Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: A group game of Axis & Allies.
2) Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: Hiking at Turkey Run State Park.
3) Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: Setting up the telescope on the main quad.
4) Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: Getting a group together for a sunday morning run.
5) Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: Organizing a campus bar crawl.

#### Summarization
1) What are objectives of the game Catan?
2) Can you summarize the basics of Chess?
3) What are the main benefits of weightlifting?
4) Can you give me a quick summary of ingredients for baking a cake?
5) Give me a quick summary of the most popular activities for a college town.

#### Prediction
1) If I do the King's Pawn Opening, what move can I expect my opponent to do?
2) I like Super Smash Brothers, what other fighting games do you think I'd enjoy?
3) How fast can my group of 8 adults do a four-mile hike with 2,000 feet of elevation gain?
4) I am a 21-year-old male and run an 18:50 5k, what is my V02 max?
5) What activities will be the most popular in Champaign-Urbana tonight?

#### Extraction
1) Review all of my favorite activities (baking, basketball, and archery) and provide estimates for the price of each one.
2) Can you translate this username in my friendslist (闇風) into english?
3) Can you give me a random list of uniquely typed activties?
4) Can you show me which of my friends have been active the last 30 days? (jerry: 1, Larry: 0, Ryan: 0, Sophie: 1, Maggie: 1)
5) How many friends do I have? (Jerry, Larry, Ryan, Sophie, Maggie)

## Define Evaluation Metrics

Metric: Accuracy (Was the answer correct?)
Metric: Clarity (Was the output understandable?)
Metric: Instruction adherence (Did the model follow the instructions?)


## Benchmarking

In [22]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM
"""
Benchmarking method, compiled from Week 6 scripts - script-3.ipynb
"""

def benchmarking(model_name, prompt, max_new_tokens=60):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    model.eval()
    start = time.time()
    inputs = tokenizer(prompt, return_tensors="pt")
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    latency = time.time() - start
    new_tokens = output.shape[1] - input_len
    text = tokenizer.decode(output[0][input_len:], skip_special_tokens=True)

    return {
        "model": model_name,
        "latency_sec": round(latency, 3),
        "new_tokens": new_tokens,
        "tokens_per_sec": round(new_tokens / latency, 2),
        "response": text
    }

In [ ]:

models = [ # ultra light
            "JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const",
          "KingNish/Reasoning-0.5b",
          "oopere/Llama-3.2-1B-pruned-40pct",

          # small
          "Alyon-AI/2_PRYMMAL-ECE-2B-SLERP-V1",
          "google/gemma-2-2b-it",
           "SpaceYL/ECE_Poirot",

         # Medium
        "OpenBuddy/openbuddy-zero-3b-v21.2-32k",
        "Qwen/Qwen2.5-1.5B-Instruct",
        "weathermanj/Menda-3b-Optim-100",

        # Large

        "grimjim/Magnolia-v3-Gemma2-8k-9B",
        "marcuscedricridia/cursa-o1-7b-v1.2-normalize-false",
        "marcuscedricridia/Abus-7B-Instruct"
]


prompts = [

    # Text generation

    ("Generate", "What are the legal moves for a knight in Chess?"),
    ("Generate", "What shoes would you recommend for a beginner runner?"),
    ("Generate", "What is the easiest sweet to bake?"),
    ("Generate", "What is the bar scene like in Champaign-Urbana?"),
    ("Generate", "What position should I play in basketball?"),

    # Classification

     ("Classification", "Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: A group game of Axis & Allies."),

     ("Classification", "Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: Hiking at Turkey Run State Park."),

     ("Classification", "Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: Setting up the telescope on the main quad."),

     ("Classification", "Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: Getting a group together for a sunday morning run."),

     ("Classification", "Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: Organizing a campus bar crawl."),

    # Summarization

     ("Summarization", "What are objectives of the game Catan?"),
     ("Summarization", "Can you summarize the basics of Chess?"),
     ("Summarization", "What are the main benefits of weightlifting?"),
     ("Summarization", "Can you give me a quick summary of ingredients for baking a cake?"),
     ("Summarization", "Give me a quick summary of the most popular activities for a college town."),

    # Prediction

    ("Prediction", "If I do the King's Pawn Opening, what move can I expect my opponent to do?"),
    ("Prediction", "I like Super Smash Brothers, what other fighting games do you think I'd enjoy?"),
    ("Prediction", "How fast can my group of 8 adults do a four-mile hike with 2,000 feet of elevation gain?"),
    ("Prediction", "I am a 21-year-old male and run an 18:50 5k, what is my V02 max?"),
    ("Prediction", "What activities will be the most popular in Champaign-Urbana tonight?"),

    # Extraction

     ("Extraction", "Review all of my favorite activities (baking, basketball, and archery) and provide estimates for the price of each one."),

     ("Extraction", "Can you translate this username in my friendslist (闇風) into english?"),

     ("Extraction", "Can you give me a random list of uniquely typed activties?"),

     ("Extraction", "Can you show me which of my friends have been active the last 30 days? (jerry: 1, Larry: 0, Ryan: 0, Sophie: 1, Maggie: 1)."),

     ("Extraction", "How many friends do I have? - Jerry, Larry, Ryan, Sophie, Maggie.")
]

for model in models:
    for prompt, p in prompts:
        result = benchmarking(model, p)
        print(model, prompt, result["tokens_per_sec"], result["latency_sec"])



JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Generate 11.96 5.017
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Generate 12.72 4.718
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Generate 12.61 4.757
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Generate 9.08 6.605
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Generate 9.09 6.599
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Classification 10.01 5.995
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Classification 12.84 4.674
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Classification 9.29 1.4
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Classification 9.35 3.959
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Classification 9.12 6.58
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Summarization 7.49 8.009
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Summarization 12.48 4.808
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Summarization 13.65 4.396
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Summarization 7.83 7.663
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Summarization 13.72 4

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Alyon-AI/2_PRYMMAL-ECE-2B-SLERP-V1 Generate 2.66 22.598


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Alyon-AI/2_PRYMMAL-ECE-2B-SLERP-V1 Generate 3.42 17.554


...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

weathermanj/Menda-3b-Optim-100 Extraction 2.53 23.743


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

weathermanj/Menda-3b-Optim-100 Extraction 2.83 12.726


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Note: my machine heavily struggles to run models larger than 10B parameters. This section does not include the massive models because it would take hours to run them. It took over an hour to run the previous cells.

# Analysis

1) It appears that text generation was the best category.
2) A larger model doesn't necessarily indicate better performance. As they grow, the time for response increases. This wouldn't be practical for our app, users expect fast response.
3) Ultimately, the KingNish/Reasoning-0.5b model has the best balance between speed/quality.

# Part 2: Cost Comparison and Infrastructure Planning

## Step 2.1: Estimate Token Usage

For the Knit activity recommendation system, we assume an average of 200 generated tokens per user request.

This estimate is reasonable because the application generates short activity recommendations rather than long-form responses.

In [2]:
avg_tokens_per_request = 200

traffic_levels = {
    "Prototype": 1_000,
    "Early Startup": 10_000,
    "Growing Product": 100_000,
    "Large Platform": 1_000_000,
    "Mass Consumer App": 10_000_000,
    "Global Platform": 100_000_000,
}

avg_tokens_per_request

200

## Step 2.2: Compute Total Token Load

The total daily token load is calculated using:

**Total Daily Token Load = Daily Active Users × Average Tokens per Request**

In [4]:
import pandas as pd

traffic_df = pd.DataFrame({
    "Daily Traffic Category": list(traffic_levels.keys()),
    "Daily Active Users": list(traffic_levels.values())
})

traffic_df["Avg Tokens per Request"] = avg_tokens_per_request
traffic_df["Total Daily Token Load"] = (
    traffic_df["Daily Active Users"] * traffic_df["Avg Tokens per Request"]
)

traffic_df

,Daily Traffic Category,Daily Active Users,Avg Tokens per Request,Total Daily Token Load
0,Prototype,1000,200,200000
1,Early Startup,10000,200,2000000
2,Growing Product,100000,200,20000000
3,Large Platform,1000000,200,200000000
4,Mass Consumer App,10000000,200,2000000000
5,Global Platform,100000000,200,20000000000


## Step 2.3: Hardware Cost Estimation (Local Hosting)

For local hosting, we estimate the infrastructure required for each of the 15 models across all six traffic levels.

We assume a GPU-enabled cloud machine with approximately 16 CPU cores and 64 GB RAM. Machine capacity and hourly cost are estimated based on model size.

In [5]:
models = [
    ("JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const", "Ultra-Light"),
    ("KingNish/Reasoning-0.5b", "Ultra-Light"),
    ("oopere/Llama-3.2-1B-pruned-40pct", "Ultra-Light"),

    ("Alyon-AI/2_PRYMMAL-ECE-2B-SLERP-V1", "Small"),
    ("google/gemma-2-2b-it", "Small"),
    ("SpaceYL/ECE_Poirot", "Small"),

    ("OpenBuddy/openbuddy-zero-3b-v21.2-32k", "Medium"),
    ("jpacifico/Distilucie-7B-Math-Instruct-DPO-v0.1", "Medium"),
    ("weathermanj/Menda-3b-Optim-100", "Medium"),

    ("grimjim/Magnolia-v3-Gemma2-8k-9B", "Large"),
    ("marcuscedricridia/cursa-o1-7b-v1.2-normalize-false", "Large"),
    ("marcuscedricridia/Abus-7B-Instruct", "Large"),

    ("MaziyarPanahi/calme-3.2-instruct-78b", "Very Large"),
    ("arcee-ai/Virtuoso-Lite", "Very Large"),
    ("Aashraf995/Gemma-Evo-10B", "Very Large"),
]

models_df = pd.DataFrame(models, columns=["Model Name", "Model Category"])
models_df

,Model Name,Model Category
0,JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const,Ultra-Light
1,KingNish/Reasoning-0.5b,Ultra-Light
2,oopere/Llama-3.2-1B-pruned-40pct,Ultra-Light
3,Alyon-AI/2_PRYMMAL-ECE-2B-SLERP-V1,Small
4,google/gemma-2-2b-it,Small
5,SpaceYL/ECE_Poirot,Small
6,OpenBuddy/openbuddy-zero-3b-v21.2-32k,Medium
7,jpacifico/Distilucie-7B-Math-Instruct-DPO-v0.1,Medium
8,weathermanj/Menda-3b-Optim-100,Medium
9,grimjim/Magnolia-v3-Gemma2-8k-9B,Large


### Infrastructure Assumptions

To estimate local hosting costs, models are grouped by size and assigned a representative GPU configuration.

Cloud pricing is based on approximate Google Cloud on-demand GPU pricing. Model throughput is treated as a planning estimate rather than a measured cloud benchmark.

In [7]:
assumptions_df = pd.DataFrame([
    ["Ultra-Light", "NVIDIA L4", 0.67, 120],
    ["Small", "NVIDIA L4", 0.67, 80],
    ["Medium", "NVIDIA A100 40GB", 3.52, 60],
    ["Large", "NVIDIA A100 40GB", 3.52, 40],
    ["Very Large", "NVIDIA A100 80GB", 4.71, 15],
], columns=[
    "Model Category",
    "GPU",
    "Hourly Cost ($)",
    "Estimated Throughput (tokens/sec)"
])

assumptions_df

,Model Category,GPU,Hourly Cost ($),Estimated Throughput (tokens/sec)
0,Ultra-Light,NVIDIA L4,0.67,120
1,Small,NVIDIA L4,0.67,80
2,Medium,NVIDIA A100 40GB,3.52,60
3,Large,NVIDIA A100 40GB,3.52,40
4,Very Large,NVIDIA A100 80GB,4.71,15


In [6]:
import math

hosting_assumptions = {
    "Ultra-Light": {
        "GPU": "NVIDIA L4",
        "Hourly Cost": 0.67,
        "Tokens per Second": 120,
    },
    "Small": {
        "GPU": "NVIDIA L4",
        "Hourly Cost": 0.67,
        "Tokens per Second": 80,
    },
    "Medium": {
        "GPU": "NVIDIA A100 40GB",
        "Hourly Cost": 3.52,
        "Tokens per Second": 60,
    },
    "Large": {
        "GPU": "NVIDIA A100 40GB",
        "Hourly Cost": 3.52,
        "Tokens per Second": 40,
    },
    "Very Large": {
        "GPU": "NVIDIA A100 80GB",
        "Hourly Cost": 4.71,
        "Tokens per Second": 15,
    },
}

hosting_assumptions

{'Ultra-Light': {'GPU': 'NVIDIA L4',
  'Hourly Cost': 0.67,
  'Tokens per Second': 120},
 'Small': {'GPU': 'NVIDIA L4', 'Hourly Cost': 0.67, 'Tokens per Second': 80},
 'Medium': {'GPU': 'NVIDIA A100 40GB',
  'Hourly Cost': 3.52,
  'Tokens per Second': 60},
 'Large': {'GPU': 'NVIDIA A100 40GB',
  'Hourly Cost': 3.52,
  'Tokens per Second': 40},
 'Very Large': {'GPU': 'NVIDIA A100 80GB',
  'Hourly Cost': 4.71,
  'Tokens per Second': 15}}

Machine capacity assumes continuous 24-hour utilization at the estimated throughput. The values are intended for comparative infrastructure planning rather than exact production pricing.

In [8]:
local_hosting_rows = []

SECONDS_PER_DAY = 86_400

for _, model in models_df.iterrows():
    category = model["Model Category"]
    assumptions = hosting_assumptions[category]

    tokens_per_second = assumptions["Tokens per Second"]
    hourly_cost = assumptions["Hourly Cost"]

    daily_capacity_per_machine = tokens_per_second * SECONDS_PER_DAY
    daily_cost_per_machine = hourly_cost * 24

    for _, traffic in traffic_df.iterrows():
        total_daily_tokens = traffic["Total Daily Token Load"]

        machines_needed = math.ceil(
            total_daily_tokens / daily_capacity_per_machine
        )

        total_machine_cost = machines_needed * daily_cost_per_machine

        local_hosting_rows.append({
            "Model Name": model["Model Name"],
            "Model Category": category,
            "Daily Traffic Category": traffic["Daily Traffic Category"],
            "Avg Tokens per User": avg_tokens_per_request,
            "Total Daily Token Load": total_daily_tokens,
            "Per Machine Cost (Daily)": round(daily_cost_per_machine, 2),
            "Machines Needed": machines_needed,
            "Total Machine Cost (Daily)": round(total_machine_cost, 2),
        })

local_hosting_df = pd.DataFrame(local_hosting_rows)

local_hosting_df

,Model Name,Model Category,Daily Traffic Category,Avg Tokens per User,Total Daily Token Load,Per Machine Cost (Daily),Machines Needed,Total Machine Cost (Daily)
0,JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const,Ultra-Light,Prototype,200,200000,16.08,1,16.08
1,JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const,Ultra-Light,Early Startup,200,2000000,16.08,1,16.08
2,JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const,Ultra-Light,Growing Product,200,20000000,16.08,2,32.16
3,JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const,Ultra-Light,Large Platform,200,200000000,16.08,20,321.60
4,JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const,Ultra-Light,Mass Consumer App,200,2000000000,16.08,193,3103.44
...,...,...,...,...,...,...,...,...
85,Aashraf995/Gemma-Evo-10B,Very Large,Early Startup,200,2000000,113.04,2,226.08
86,Aashraf995/Gemma-Evo-10B,Very Large,Growing Product,200,20000000,113.04,16,1808.64
87,Aashraf995/Gemma-Evo-10B,Very Large,Large Platform,200,200000000,113.04,155,17521.20
88,Aashraf995/Gemma-Evo-10B,Very Large,Mass Consumer App,200,2000000000,113.04,1544,174533.76


## Step 2.4: Hugging Face Inference API Cost Comparison

For the API comparison, each local model category is mapped to a similarly sized paid model available through Hugging Face Inference Providers.

Because our token estimate represents generated output tokens, the calculation uses the provider's output-token price.

API prices are expressed in dollars per 1 million output tokens. These prices are used as planning estimates and may change over time.

In [9]:
api_assumptions = {
    "Ultra-Light": {
        "Comparable Paid Model": "Qwen/Qwen2.5-Coder-3B-Instruct",
        "Output Cost per 1M Tokens": 0.03,
    },
    "Small": {
        "Comparable Paid Model": "Qwen/Qwen3-4B-Instruct-2507",
        "Output Cost per 1M Tokens": 0.03,
    },
    "Medium": {
        "Comparable Paid Model": "Qwen/Qwen3-8B",
        "Output Cost per 1M Tokens": 0.18,
    },
    "Large": {
        "Comparable Paid Model": "Qwen/Qwen3-14B",
        "Output Cost per 1M Tokens": 0.20,
    },
    "Very Large": {
        "Comparable Paid Model": "Qwen/Qwen2.5-72B-Instruct",
        "Output Cost per 1M Tokens": 0.40,
    },
}

api_assumptions_df = pd.DataFrame([
    {
        "Local Model Category": category,
        "Comparable Paid Model": values["Comparable Paid Model"],
        "Output Cost ($/1M Tokens)": values["Output Cost per 1M Tokens"],
    }
    for category, values in api_assumptions.items()
])

api_assumptions_df

,Local Model Category,Comparable Paid Model,Output Cost ($/1M Tokens)
0,Ultra-Light,Qwen/Qwen2.5-Coder-3B-Instruct,0.03
1,Small,Qwen/Qwen3-4B-Instruct-2507,0.03
2,Medium,Qwen/Qwen3-8B,0.18
3,Large,Qwen/Qwen3-14B,0.20
4,Very Large,Qwen/Qwen2.5-72B-Instruct,0.40


In [10]:
api_cost_rows = []

API_SERVER_DAU_CAPACITY = 100_000
API_SERVER_HOURLY_COST = 0.10
API_SERVER_DAILY_COST = API_SERVER_HOURLY_COST * 24

for _, model in models_df.iterrows():
    category = model["Model Category"]
    api_info = api_assumptions[category]

    comparable_model = api_info["Comparable Paid Model"]
    cost_per_million = api_info["Output Cost per 1M Tokens"]
    cost_per_token = cost_per_million / 1_000_000

    for _, traffic in traffic_df.iterrows():
        total_tokens = traffic["Total Daily Token Load"]
        daily_users = traffic["Daily Active Users"]

        estimated_api_cost = total_tokens * cost_per_token

        api_machines_needed = math.ceil(
            daily_users / API_SERVER_DAU_CAPACITY
        )

        api_infrastructure_cost = (
            api_machines_needed * API_SERVER_DAILY_COST
        )

        total_cost = estimated_api_cost + api_infrastructure_cost

        api_cost_rows.append({
            "Local LLM Model": model["Model Name"],
            "Local Model Category": category,
            "Comparable Paid Model": comparable_model,
            "Cost per Token ($)": cost_per_token,
            "Daily Traffic Category": traffic["Daily Traffic Category"],
            "Total Token Load": total_tokens,
            "Estimated API Cost (Daily)": round(estimated_api_cost, 2),
            "Machines Required (API Infrastructure)": api_machines_needed,
            "API Infrastructure Cost (Daily)": round(api_infrastructure_cost, 2),
            "Total Cost (Daily)": round(total_cost, 2),
        })

api_cost_df = pd.DataFrame(api_cost_rows)

api_cost_df

,Local LLM Model,Local Model Category,Comparable Paid Model,Cost per Token ($),Daily Traffic Category,Total Token Load,Estimated API Cost (Daily),Machines Required (API Infrastructure),API Infrastructure Cost (Daily),Total Cost (Daily)
0,JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const,Ultra-Light,Qwen/Qwen2.5-Coder-3B-Instruct,3.000000e-08,Prototype,200000,0.01,1,2.4,2.41
1,JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const,Ultra-Light,Qwen/Qwen2.5-Coder-3B-Instruct,3.000000e-08,Early Startup,2000000,0.06,1,2.4,2.46
2,JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const,Ultra-Light,Qwen/Qwen2.5-Coder-3B-Instruct,3.000000e-08,Growing Product,20000000,0.60,1,2.4,3.00
3,JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const,Ultra-Light,Qwen/Qwen2.5-Coder-3B-Instruct,3.000000e-08,Large Platform,200000000,6.00,10,24.0,30.00
4,JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const,Ultra-Light,Qwen/Qwen2.5-Coder-3B-Instruct,3.000000e-08,Mass Consumer App,2000000000,60.00,100,240.0,300.00
...,...,...,...,...,...,...,...,...,...,...
85,Aashraf995/Gemma-Evo-10B,Very Large,Qwen/Qwen2.5-72B-Instruct,4.000000e-07,Early Startup,2000000,0.80,1,2.4,3.20
86,Aashraf995/Gemma-Evo-10B,Very Large,Qwen/Qwen2.5-72B-Instruct,4.000000e-07,Growing Product,20000000,8.00,1,2.4,10.40
87,Aashraf995/Gemma-Evo-10B,Very Large,Qwen/Qwen2.5-72B-Instruct,4.000000e-07,Large Platform,200000000,80.00,10,24.0,104.00
88,Aashraf995/Gemma-Evo-10B,Very Large,Qwen/Qwen2.5-72B-Instruct,4.000000e-07,Mass Consumer App,2000000000,800.00,100,240.0,1040.00


## Step 2.5: Analysis

To determine when local hosting or API inference is more cost-effective, we compare the daily cost of both approaches for each model category and traffic level.

In [11]:
cost_comparison_df = local_hosting_df.merge(
    api_cost_df,
    left_on=[
        "Model Name",
        "Model Category",
        "Daily Traffic Category"
    ],
    right_on=[
        "Local LLM Model",
        "Local Model Category",
        "Daily Traffic Category"
    ]
)

cost_comparison_df["Cheaper Option"] = cost_comparison_df.apply(
    lambda row: (
        "Local Hosting"
        if row["Total Machine Cost (Daily)"] < row["Total Cost (Daily)"]
        else "API"
    ),
    axis=1
)

comparison_summary = cost_comparison_df[
    [
        "Model Name",
        "Model Category",
        "Daily Traffic Category",
        "Total Machine Cost (Daily)",
        "Total Cost (Daily)",
        "Cheaper Option",
    ]
]

comparison_summary

,Model Name,Model Category,Daily Traffic Category,Total Machine Cost (Daily),Total Cost (Daily),Cheaper Option
0,JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const,Ultra-Light,Prototype,16.08,2.41,API
1,JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const,Ultra-Light,Early Startup,16.08,2.46,API
2,JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const,Ultra-Light,Growing Product,32.16,3.00,API
3,JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const,Ultra-Light,Large Platform,321.60,30.00,API
4,JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const,Ultra-Light,Mass Consumer App,3103.44,300.00,API
...,...,...,...,...,...,...
85,Aashraf995/Gemma-Evo-10B,Very Large,Early Startup,226.08,3.20,API
86,Aashraf995/Gemma-Evo-10B,Very Large,Growing Product,1808.64,10.40,API
87,Aashraf995/Gemma-Evo-10B,Very Large,Large Platform,17521.20,104.00,API
88,Aashraf995/Gemma-Evo-10B,Very Large,Mass Consumer App,174533.76,1040.00,API


In [12]:
summary_by_traffic = (
    comparison_summary
    .groupby(["Daily Traffic Category", "Cheaper Option"])
    .size()
    .unstack(fill_value=0)
)

summary_by_traffic

Cheaper Option,API
Daily Traffic Category,
Early Startup,15
Global Platform,15
Growing Product,15
Large Platform,15
Mass Consumer App,15
Prototype,15


## Step 2.5: Analysis

1. **At what scale is local hosting cheaper?**

   Under the assumptions used in this analysis, local hosting is not cheaper at any of the six traffic scales. For all 15 models and all traffic categories, the estimated daily API cost is lower than the corresponding local GPU hosting cost. The main reason is that local hosting requires dedicated GPU machines to run continuously, while API pricing scales directly with actual token usage.

2. **At what scale is API usage cheaper?**

   API usage is cheaper at every tested scale, from the Prototype level with 1,000 daily active users to the Global Platform level with 100 million daily active users. This result is especially clear for larger models, where local hosting requires expensive A100-class GPUs and a rapidly increasing number of machines as token demand grows.

3. **When would you switch your architecture?**

   Based strictly on this cost model, we would continue using API-based inference rather than switching to full local hosting. However, a switch to local hosting could become reasonable if API token prices increased significantly, GPU utilization became much more efficient, discounted or owned hardware was available, or privacy and data-control requirements made external APIs unsuitable. A hybrid architecture could also be useful, with smaller local models handling simple requests and API models handling more complex requests.